<a href="https://colab.research.google.com/github/replysantosh-lang/ECARAgenticAI/blob/main/01_Building_Your_First_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install requests==2.32.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 2.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.34.2
    Uninstalling requests-2.34.2:
      Successfully uninstalled requests-2.34.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.2 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.


In [1]:
# ============================================================
# Building Your First RAG: PDF Q&A System
# Platform: Google Colab
# ============================================================

!pip install -q langchain langchain-community langchain-openai faiss-cpu pypdf tiktoken

import os
from google.colab import files, userdata

# ------------------------------------------------------------
# 1. Set OpenAI API Key
# ------------------------------------------------------------

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [7]:
# ------------------------------------------------------------
# 2. Upload PDF
# ------------------------------------------------------------

uploaded = files.upload()

# Get the original filename and content from the uploaded dictionary
original_filename = list(uploaded.keys())[0]
file_content = uploaded[original_filename]

# Define a new, simpler filename
new_filename = 'document.pdf'
new_file_path = os.path.join('/content', new_filename)

# Write the content to the new file
with open(new_file_path, 'wb') as f:
    f.write(file_content)

pdf_path = new_filename
print(f"Original PDF '{original_filename}' uploaded and saved as '{new_filename}'")

Saving a-practical-guide-to-building-agents.pdf to a-practical-guide-to-building-agents (1).pdf
Original PDF 'a-practical-guide-to-building-agents (1).pdf' uploaded and saved as 'document.pdf'


In [8]:
# Rename the uploaded file to a simpler name to avoid issues with special characters
old_path = os.path.join('/content', pdf_path)
new_file_name = 'document.pdf'
new_path = os.path.join('/content', new_file_name)

os.rename(old_path, new_path)
pdf_path = new_file_name

print(f"Renamed '{old_path}' to '{new_path}'")

Renamed '/content/document.pdf' to '/content/document.pdf'


In [9]:
import os
# ------------------------------------------------------------
# 3. Load PDF
# ------------------------------------------------------------

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(os.path.join('/content', pdf_path))
documents = loader.load()

print("Total pages loaded:", len(documents))
print(documents[0].page_content[:500])

Total pages loaded: 34
A practical  
guide to  
building agents


In [10]:
# ------------------------------------------------------------
# 4. Split PDF Text into Chunks
# ------------------------------------------------------------

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Total chunks created:", len(chunks))
print(chunks[0].page_content[:500])

Total chunks created: 46
A practical  
guide to  
building agents


In [11]:
# ------------------------------------------------------------
# 5. Create Embeddings and Vector Store
# ------------------------------------------------------------

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

print("Vector database created successfully.")

Vector database created successfully.


In [12]:
# ------------------------------------------------------------
# 6. Create RAG Chain
# ------------------------------------------------------------

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

prompt = ChatPromptTemplate.from_template("""
You are a helpful PDF Q&A assistant.

Answer the question using only the context provided.
If the answer is not present in the context, say:
"I could not find this information in the PDF."

Context:
{context}

Question:
{question}

Answer:
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [13]:
# ------------------------------------------------------------
# 7. Ask Questions
# ------------------------------------------------------------

question = "Summarize the key points of this PDF."

answer = rag_chain.invoke(question)

print("Question:", question)
print("\nAnswer:")
print(answer)

Question: Summarize the key points of this PDF.

Answer:
The PDF provides a practical guide to building agents, covering the following key points:

1. Definition of an agent and when to build one.
2. Foundations of agent design.
3. Importance of guardrails, including:
   - Rules-based protections to prevent known threats.
   - Output validation to ensure responses align with brand values.
4. Recommendations for building guardrails:
   - Focus on data privacy and content safety.
   - Add guardrails based on real-world edge cases and failures.
   - Optimize for security and user experience as the agent evolves.


In [14]:
# ------------------------------------------------------------
# 8. Ask Custom Questions Interactively
# ------------------------------------------------------------

while True:
    user_question = input("\nAsk a question about the PDF, or type 'exit': ")

    if user_question.lower() == "exit":
        break

    response = rag_chain.invoke(user_question)

    print("\nAnswer:")
    print(response)


Ask a question about the PDF, or type 'exit': What is an agent?

Answer:
An agent is a system that independently accomplishes tasks on your behalf, performing workflows with a high degree of independence. It leverages a large language model (LLM) to manage workflow execution and make decisions, recognizing when a workflow is complete and proactively correcting its actions if needed.

Ask a question about the PDF, or type 'exit': exit


In [15]:
# ------------------------------------------------------------
# 9. Show Source Chunks Used
# ------------------------------------------------------------

question = "What are the main topics discussed in this document?"

source_docs = retriever.invoke(question)

print("Retrieved Source Chunks:\n")

for i, doc in enumerate(source_docs, start=1):
    print(f"--- Source Chunk {i} ---")
    print("Page:", doc.metadata.get("page"))
    print(doc.page_content[:700])
    print()

Retrieved Source Chunks:

--- Source Chunk 1 ---
Page: 26
Rules-based protections Simple deterministic measures (blocklists, input length limits, 
regex filters) to prevent known threats like prohibited terms or 
SQL injections.
Output validation Ensures responses align with brand values via prompt 
engineering and content checks, preventing outputs that  
could harm your brand’s integrity.
Building guardrails
Set up guardrails that address the risks you’ve already identified for your use case and layer in 
additional ones as you uncover new vulnerabilities.  
We’ve found the following heuristic to be effective:
01 Focus on data privacy and content safety
02 Add new guardrails based on real-world edge cases and failures you encounter
03 Optimize fo

--- Source Chunk 2 ---
Page: 11
Y ou can use advanced models, like o1 or o3-mini, to automatically generate instructions from 
existing documents. Here’s a sample prompt illustrating this approach:
Unset
1 “You are an expert in writing inst

In [16]:
# ------------------------------------------------------------
# 10. Final RAG Function with Answer + Sources
# ------------------------------------------------------------

def ask_pdf(question):
    source_docs = retriever.invoke(question)

    context = format_docs(source_docs)

    final_prompt = prompt.invoke({
        "context": context,
        "question": question
    })

    answer = llm.invoke(final_prompt).content

    print("Question:")
    print(question)

    print("\nAnswer:")
    print(answer)

    print("\nSources:")
    for i, doc in enumerate(source_docs, start=1):
        print(f"\nSource {i}")
        print("Page:", doc.metadata.get("page"))
        print(doc.page_content[:500])

ask_pdf("What is this document about?")

Question:
What is this document about?

Answer:
This document is a practical guide to building agents, focusing on creating high-quality instructions for LLM-powered applications. It emphasizes the importance of clear instructions to reduce ambiguity, improve decision-making, and ensure smooth workflow execution. The guide also provides best practices for configuring instructions, such as using existing documents, breaking down tasks into smaller steps, and defining clear actions for agents.

Sources:

Source 1
Page: 11
Y ou can use advanced models, like o1 or o3-mini, to automatically generate instructions from 
existing documents. Here’s a sample prompt illustrating this approach:
Unset
1 “You are an expert in writing instructions for an LLM agent. Convert the 
following help center document into a clear set of instructions, written in 
a numbered list. The document will be a policy followed by an LLM. Ensure 
that there is no ambiguity, and that the instructions are written as 
dire